<a href="https://colab.research.google.com/github/Gloria888520/Gloria/blob/main/Weibo_Narrative_Project111.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import files

uploaded = files.upload()

Saving Gloria_Attempt2_Project_Package.zip to Gloria_Attempt2_Project_Package (2).zip


In [14]:
import io
import os
import zipfile

filename = next(iter(uploaded))

with zipfile.ZipFile(io.BytesIO(uploaded[filename])) as z:
    z.extractall("/content/weibo_project")

os.chdir("/content/weibo_project/course_repo")

print("Project File is ready")

Project File is ready


In [15]:
import pandas as pd

df = pd.read_csv("data/narrative_summaries.csv")
splits = pd.read_csv("results/split_assignments.csv")

training_rows = df.merge(
    splits[["record_id", "split"]],
    on="record_id"
)
training_rows = training_rows[
    training_rows["split"] == "train"
]

print("完整数据的行数、列数：", df.shape)

display(
    training_rows[
        ["summary", "highest_score_ecosystem"]
    ].head(20)
)

完整数据的行数、列数： (2147, 14)


,summary,highest_score_ecosystem
2,"US denies allegations of biological warfare in Ukraine, despite evidence contradicting claims.",Russia
3,Ukraine and Russia negotiations are unlikely to yield results due to fundamental disagreements.,Russia
4,"China maintains a neutral stance on the Ukraine-Russia conflict, with some citizens supporting Russia and others Ukraine.",Ukraine
5,"Russian President Putin escalates military action in Ukraine, testing Western resolve.",Ukraine
6,The US is manipulating Ukraine to create chaos in Europe and distract from its rivalry with China.,Russia
7,"Ukraine's pursuit of NATO membership sparks conflict with Russia, prompting concerns about regional stability.",Russia
8,Weibo may display users' real-time locations in comments about Ukraine or Russia.,Russia
10,Ukraine's fate is largely self-inflicted due to its blind trust in Western powers.,Ukraine
11,US media suggests China may face sanctions if Russia invades Ukraine.,Russia
15,"China supports Russia amid tensions with the US, viewing similar suppression.",China


In [16]:
pd.set_option("display.max_colwidth", None)

display(
    training_rows[
        ["summary", "highest_score_ecosystem"]
    ].head(20)
)

,summary,highest_score_ecosystem
2,"US denies allegations of biological warfare in Ukraine, despite evidence contradicting claims.",Russia
3,Ukraine and Russia negotiations are unlikely to yield results due to fundamental disagreements.,Russia
4,"China maintains a neutral stance on the Ukraine-Russia conflict, with some citizens supporting Russia and others Ukraine.",Ukraine
5,"Russian President Putin escalates military action in Ukraine, testing Western resolve.",Ukraine
6,The US is manipulating Ukraine to create chaos in Europe and distract from its rivalry with China.,Russia
7,"Ukraine's pursuit of NATO membership sparks conflict with Russia, prompting concerns about regional stability.",Russia
8,Weibo may display users' real-time locations in comments about Ukraine or Russia.,Russia
10,Ukraine's fate is largely self-inflicted due to its blind trust in Western powers.,Ukraine
11,US media suggests China may face sanctions if Russia invades Ukraine.,Russia
15,"China supports Russia amid tensions with the US, viewing similar suppression.",China


In [17]:
# 检查关键字段
required = [
    "summary",
    "pmi_china",
    "pmi_us",
    "pmi_russia",
    "pmi_ukraine",
    "highest_score_ecosystem"
]

assert len(df) == 2147
assert df["record_id"].is_unique
assert df[required].notna().all().all()

# 查看类别数量
print("类别数量：")
print(df["highest_score_ecosystem"].value_counts())

# 查看训练、验证、测试划分
print("\n数据划分：")
print(splits["split"].value_counts())

# 同一个摘要组不能出现在多个集合中
groups_per_split = (
    splits.groupby("summary_group_id")["split"].nunique()
)
assert groups_per_split.max() == 1

print("\nPASS：基本数据检查通过")

类别数量：
highest_score_ecosystem
Ukraine      829
Russia       808
China        411
US            85
ambiguous     14
Name: count, dtype: int64

数据划分：
split
train            1279
test_reserved     427
validation        427
Name: count, dtype: int64

PASS：基本数据检查通过


In [18]:
import os
import sys
import subprocess

os.chdir("/content/weibo_project/course_repo")

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", "requirements.txt"],
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', 'requirements.txt'], returncode=0)

In [19]:
env = os.environ.copy()
env["OPENBLAS_NUM_THREADS"] = "1"
env["OMP_NUM_THREADS"] = "1"

# 训练模型，计算验证集成绩
subprocess.run(
    [sys.executable, "pilot.py",
     "--out", "results_rerun"],
    check=True,
    env=env
)

# 检查是否复现已有结果
subprocess.run(
    [sys.executable, "check_pilot.py",
     "--out", "results_rerun"],
    check=True
)

CompletedProcess(args=['/usr/bin/python3', 'check_pilot.py', '--out', 'results_rerun'], returncode=0)